In [ ]:
from datetime import datetime, timedelta

In [ ]:
import time
import re
import pandas as pd
from datetime import date

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait


# =============================
# 기본 설정
# =============================

START_DATE = datetime(2026, 5, 22, 00, 00,0)
END_DATE = datetime(2026, 5, 27, 23, 59,0)

MATCH_38R = datetime(2026, 5, 25, 0, 0, 0)

BASE_YEAR = 2026

# 100개에서 터진다고 했으니까 70개마다 새로고침 추천
RESTART_EVERY = 70

MAX_CHECK_POSTS = 2000

SAVE_FILE_CSV = "liverpool_cafe_38R_before_after.csv"
SAVE_FILE_XLSX = "liverpool_cafe_38R_before_after.xlsx"


# =============================
# 토트넘 카페 코드와 컬럼 통일용 함수
# 수집 방식은 건드리지 않고, 저장 형식만 맞춤
# =============================

FINAL_COLUMNS = [
    "source", "round", "title", "post_date", "comment_date",
    "text_original", "text_clean", "content_type", "period", "url"
]

def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_period(dt):
    if pd.isna(dt):
        return "unknown"

    if dt < MATCH_38R:
        return "before_38R"
    else:
        return "after_38R"

def parse_cafe_date(date_text):
    if date_text is None:
        return None

    text = str(date_text).strip()
    text = re.sub(r"\s+", " ", text)

    # -----------------------------
    # 1. 연도 + 날짜 + 시간
    # 예: 2026.05.25. 23:43
    # -----------------------------
    m = re.search(
        r"(\d{4})\.(\d{1,2})\.(\d{1,2})\.?\s+(\d{1,2}):(\d{2})",
        text
    )

    if m:
        y = int(m.group(1))
        mo = int(m.group(2))
        d = int(m.group(3))
        h = int(m.group(4))
        mi = int(m.group(5))

        return datetime(y, mo, d, h, mi, 0)

    # -----------------------------
    # 2. 월일 + 시간
    # 예: 05.25. 23:43
    # -----------------------------
    m = re.search(
        r"(\d{1,2})\.(\d{1,2})\.?\s+(\d{1,2}):(\d{2})",
        text
    )

    if m:
        mo = int(m.group(1))
        d = int(m.group(2))
        h = int(m.group(3))
        mi = int(m.group(4))

        return datetime(BASE_YEAR, mo, d, h, mi, 0)

    # -----------------------------
    # 3. 연도 + 날짜만 있음
    # 예: 2026.05.25.
    # 시간이 없으므로 00:00:00 처리됨
    # -----------------------------
    m = re.search(
        r"(\d{4})\.(\d{1,2})\.(\d{1,2})\.?",
        text
    )

    if m:
        y = int(m.group(1))
        mo = int(m.group(2))
        d = int(m.group(3))

        return datetime(y, mo, d, 0, 0, 0)

    # -----------------------------
    # 4. 월일만 있음
    # 예: 05.25.
    # -----------------------------
    m = re.search(
        r"(\d{1,2})\.(\d{1,2})\.?",
        text
    )

    if m:
        mo = int(m.group(1))
        d = int(m.group(2))

        return datetime(BASE_YEAR, mo, d, 0, 0, 0)

    return None


# 기존 코드 호환용
parse_article_date = parse_cafe_date


def get_article_title():
    global driver

    selectors = [
        "h3.title_text",
        ".title_text",
        ".ArticleTitle",
        ".article_header .title",
        "h3"
    ]

    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                title = elem.text.strip()
                if title:
                    return title
        except:
            pass

    return ""


# =============================
# 드라이버 생성 함수
# =============================

def create_driver():
    options = webdriver.ChromeOptions()

    options.add_argument("--start-maximized")

    # 메모리 줄이기
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-dev-shm-usage")

    

    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)

    return driver


# =============================
# cafe_main iframe 들어가기
# =============================

def enter_cafe_main():
    global driver

    try:
        driver.switch_to.default_content()
        driver.switch_to.frame("cafe_main")
        print("cafe_main iframe 진입 완료")
        return True

    except Exception as e:
        print("cafe_main iframe 진입 실패:", e)
        return False


def get_real_article_url():
    global driver

    try:
        # 현재 iframe 안의 실제 URL 가져오기
        real_url = driver.execute_script("return window.location.href;")
        return real_url

    except Exception as e:
        print("실제 글 URL 가져오기 실패:", e)
        return driver.current_url



# =============================
# 날짜 파싱 함수
# =============================

def parse_article_date(date_text):
    if not date_text:
        return None

    date_text = date_text.strip()

    # 예: 2026.05.25. 23:43
    m = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})\.?\s*(\d{1,2}):(\d{2})", date_text)
    if m:
        y = int(m.group(1))
        mo = int(m.group(2))
        d = int(m.group(3))
        h = int(m.group(4))
        mi = int(m.group(5))
        return datetime(y, mo, d, h, mi)

    # 예: 05.25. 23:43
    m = re.search(r"(\d{1,2})\.(\d{1,2})\.?\s*(\d{1,2}):(\d{2})", date_text)
    if m:
        mo = int(m.group(1))
        d = int(m.group(2))
        h = int(m.group(3))
        mi = int(m.group(4))
        return datetime(2026, mo, d, h, mi)

    # 예: 2026.05.25. 시간 없는 경우
    m = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})", date_text)
    if m:
        y = int(m.group(1))
        mo = int(m.group(2))
        d = int(m.group(3))
        return datetime(y, mo, d, 0, 0)

    # 예: 05.25. 시간 없는 경우
    m = re.search(r"(\d{1,2})\.(\d{1,2})", date_text)
    if m:
        mo = int(m.group(1))
        d = int(m.group(2))
        return datetime(2026, mo, d, 0, 0)

    return None


# =============================
# 날짜 가져오기
# =============================

def get_article_date():
    global driver

    selectors = [
        ".date",
        "span.date",
        ".ArticleHeader .date",
        ".article_info .date",
        ".WriterInfo .date"
    ]

    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)

            for elem in elems:
                date_text = elem.text.strip()
                article_date = parse_article_date(date_text)

                if article_date is not None:
                    return date_text, article_date

        except:
            pass

    return "", None


# =============================
# 본문 수집
# 네 기존 코드 거의 그대로
# =============================

def get_article_content():
    global driver

    try:
        tmp = driver.find_elements(By.CSS_SELECTOR, '.se-text-paragraph')

        paragraph = []
        
        for x in tmp:
            text = x.text.strip()
            if text:
                paragraph.append(text)
        
        content = "\n".join(paragraph)

        print("본문 길이:", len(content))
        print("본문 미리보기:", content[:100])

    except Exception as e:
        print("본문 수집 실패:", e)
        content = ""

    return content


# =============================
# 다음글 클릭
# 네 기존 방식 유지
# =============================

def get_next_article_url():
    global driver

    try:
        next_btn = driver.find_element(By.PARTIAL_LINK_TEXT, "다음글")

        next_url = next_btn.get_attribute("href")

        print("다음 버튼 텍스트:", next_btn.text)
        print("다음글 URL:", next_url)

        if next_url is None or next_url == "":
            print("다음글 href가 비어 있음")
            return None

        return next_url

    except Exception as e:
        print("다음글 URL 가져오기 실패:", e)
        return None


# =============================
# 추천 방식: soft restart
# 창은 안 끄고 현재 URL 다시 로딩
# =============================

def soft_restart(article_url):
    global driver

    print("\n===================================")
    print(f"{RESTART_EVERY}개 확인 완료 → 현재 article_url로 다시 로딩")
    print("재접속 URL:", article_url)
    print("===================================\n")

    try:
        driver.switch_to.default_content()
    except:
        pass

    driver.get(article_url)
    time.sleep(5)

    enter_cafe_main()

    print("soft restart 완료")

def get_next_article_url():
    global driver

    try:
        # cafe_main iframe 안에 있는 상태라고 가정
        links = driver.find_elements(By.TAG_NAME, "a")

        for link in links:
            text = link.text.strip()
            href = link.get_attribute("href")

            if "다음글" in text:
                print("찾은 다음글 텍스트:", text)
                print("찾은 다음글 href:", href)

                if href:
                    return href

        print("다음글 링크를 못 찾음")
        return None

    except Exception as e:
        print("다음글 URL 가져오기 실패:", e)
        return None

# =============================
# 댓글 날짜 파싱
# 기존 parse_article_date 재사용 가능
# =============================

def parse_comment_date(date_text):
    """
    네이버 카페 댓글 날짜 예시:
    2026.05.25. 13:20
    05.25. 13:20
    """
    return parse_article_date(date_text)


# =============================
# 댓글 더보기 클릭
# 댓글이 접혀있거나 더보기 버튼이 있을 때 최대한 펼침
# =============================

# def click_comment_more_buttons(max_click=10):
#     global driver

#     for _ in range(max_click):
#         clicked = False

#         try:
#             buttons = driver.find_elements(By.TAG_NAME, "a")

#             for btn in buttons:
#                 text = btn.text.strip()

#                 if "댓글 더보기" in text or "더보기" in text:
#                     try:
#                         driver.execute_script("arguments[0].click();", btn)
#                         time.sleep(1)
#                         clicked = True
#                         print("댓글 더보기 클릭")
#                         break
#                     except:
#                         pass

#         except Exception as e:
#             print("댓글 더보기 확인 실패:", e)
#             break

#         if not clicked:
#             break


# =============================
# 댓글 수집 함수
# 본문 수집 방식은 건드리지 않고 댓글만 추가 수집
# =============================

def get_article_comments():
    global driver

    comments = []

    # 혹시 댓글이 아래쪽 로딩 방식이면 살짝 아래로 스크롤
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
    except:
        pass

    # 댓글 후보 selector를 넓게 잡음
    comment_item_selectors = [
        "li.CommentItem",
        ".CommentItem",
        ".comment_item",
        ".CommentBox",
        ".comment_box",
        ".u_cbox_comment",
        "li[class*='Comment']",
        "div[class*='CommentItem']",
        "div[class*='comment']"
    ]

    comment_items = []

    for selector in comment_item_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)

            # 너무 큰 영역 하나만 잡히는 경우를 줄이기 위해 text 있는 것만 필터링
            elems = [e for e in elems if e.text.strip()]

            if elems:
                comment_items = elems
                print(f"댓글 후보 영역 발견 selector: {selector}, 개수: {len(comment_items)}")
                break

        except Exception as e:
            pass

    if not comment_items:
        print("댓글 없음 또는 댓글 영역 못 찾음")

        # 디버깅용: 페이지 안에 댓글 관련 텍스트가 있는지 확인
        try:
            body_text = driver.find_element(By.TAG_NAME, "body").text
            print("body에 '댓글' 포함 여부:", "댓글" in body_text)
            print("body 텍스트 일부:")
            print(body_text[-500:])
        except:
            pass

        return comments

    for item in comment_items:
        raw = item.text.strip()

        if not raw:
            continue

        comment_text = ""
        comment_date_text = ""
        comment_date = None

        # -----------------------------
        # 1. 댓글 내용 가져오기
        # -----------------------------
        text_selectors = [
            ".text_comment",
            "span.text_comment",
            ".comment_text_box",
            ".comment_text_view",
            ".u_cbox_contents",
            "span[class*='text']",
            "div[class*='text']"
        ]

        for selector in text_selectors:
            try:
                elems = item.find_elements(By.CSS_SELECTOR, selector)

                for elem in elems:
                    text = elem.text.strip()

                    if text:
                        comment_text = text
                        break

                if comment_text:
                    break

            except:
                pass

        # selector로 댓글 내용 못 찾으면 item 전체 텍스트에서 날짜/작성자 제외하고 대충 사용
        if not comment_text:
            lines = [x.strip() for x in raw.split("\n") if x.strip()]

            # 보통 댓글 item 안에는 작성자, 내용, 날짜가 섞임
            # 일단 날짜처럼 보이는 줄은 제외
            filtered_lines = []
            for line in lines:
                if re.search(r"\d{4}\.\d{1,2}\.\d{1,2}", line):
                    continue
                if re.search(r"\d{1,2}\.\d{1,2}\.", line):
                    continue
                if line in ["답글", "신고", "삭제", "수정"]:
                    continue

                filtered_lines.append(line)

            if filtered_lines:
                # 작성자까지 섞일 수 있어서 보통 두 번째 줄 이후가 댓글일 가능성이 높음
                if len(filtered_lines) >= 2:
                    comment_text = " ".join(filtered_lines[1:])
                else:
                    comment_text = filtered_lines[0]

        # -----------------------------
        # 2. 댓글 날짜 가져오기
        # -----------------------------
        date_selectors = [
            ".comment_info_date",
            "span.comment_info_date",
            ".date",
            "span.date",
            ".u_cbox_date",
            "span[class*='date']",
            "em[class*='date']"
        ]

        for selector in date_selectors:
            try:
                elems = item.find_elements(By.CSS_SELECTOR, selector)

                for elem in elems:
                    date_text = elem.text.strip()
                    parsed_date = parse_comment_date(date_text)

                    if parsed_date is not None:
                        comment_date_text = date_text
                        comment_date = parsed_date
                        break

                if comment_date is not None:
                    break

            except:
                pass

        # selector로 날짜 못 찾으면 raw 텍스트에서 날짜 패턴 검색
        if comment_date is None:
            m = re.search(r"(\d{4}\.\d{1,2}\.\d{1,2}\.?\s*\d{1,2}:\d{2})", raw)
            if not m:
                m = re.search(r"(\d{1,2}\.\d{1,2}\.?\s*\d{1,2}:\d{2})", raw)

            if m:
                comment_date_text = m.group(1)
                comment_date = parse_comment_date(comment_date_text)

        # -----------------------------
        # 3. 저장
        # -----------------------------
        if comment_text:
            comments.append({
                "comment_text": comment_text,
                "comment_date_text": comment_date_text,
                "comment_date": comment_date
            })

    print("수집된 댓글 수:", len(comments))

    return comments

In [ ]:
driver = create_driver()
wait = WebDriverWait(driver, 10)

driver.get("https://nid.naver.com/nidlogin.login")

print("네이버 로그인하세요.")
print("로그인 후 수집 시작할 카페 글 페이지까지 직접 이동하세요.")
print("그 다음 3번 실행 코드를 돌리면 됩니다.")

In [ ]:
enter_cafe_main()

In [ ]:
# =============================
# 다시 시작할 글 URL 직접 지정
# 처음 시작할 때는 시작 글 URL 넣기
# 중간부터 다시 시작하고 싶으면 여기에 그 글 URL을 직접 넣기
# =============================

article_url = "https://cafe.naver.com/f-e/cafes/22321016/articles/921034?boardtype=L&menuid=24&referrerAllArticles=false&page=47"


# =============================
# posts 유지
# 셀 다시 실행해도 기존 posts 날아가지 않게 함
# =============================

try:
    posts
except NameError:
    posts = []

try:
    checked_count
except NameError:
    checked_count = 0

try:
    saved_count
except NameError:
    saved_count = len(posts)


# =============================
# 수집 시작
# =============================

for i in range(checked_count, MAX_CHECK_POSTS):

    print(f"\n===== {i+1}번째 글 수집 시작 =====")

    print("현재 article_url:", article_url)

    # 현재 article_url로 직접 이동
    driver.switch_to.default_content()
    driver.get(article_url)
    time.sleep(3)

    # iframe 진입
    enter_cafe_main()

    # 70개마다 창 유지한 채 현재 글 다시 로딩
    if i > 0 and i % RESTART_EVERY == 0:
        soft_restart(article_url)
        time.sleep(2)
        enter_cafe_main()

    # -----------------------------
    # 1. 날짜 수집
    # -----------------------------




    try:
        date_text, article_date = get_article_date()

        print("날짜 원문:", date_text)
        print("파싱 날짜:", article_date)

    except Exception as e:
        print("날짜 수집 실패:", e)
        date_text = ""
        article_date = None

    # -----------------------------
    # 1-1. 제목 수집
    # -----------------------------
    try:
        title = get_article_title()
        print("제목:", title)
    except Exception as e:
        print("제목 수집 실패:", e)
        title = ""

    # -----------------------------
    # 2. 날짜 기준 필터링
    # -----------------------------

    if article_date is None:
        print("날짜 확인 실패 → 저장 안 함")

    elif article_date < START_DATE:
        print("START_DATE보다 이전 글 도달 → 수집 종료")
        break

    elif START_DATE <= article_date <= END_DATE:

        # -----------------------------
        # 3. 본문 수집
        # -----------------------------
        content = get_article_content()

        posts.append({
            "source": "cafe",
            "round": "38R_before_after",
            "title": title,
            "post_date": article_date,
            "comment_date": None,
            "text_original": content,
            "text_clean": clean_text(content),
            "content_type": "cafe_post",
            "period": get_period(article_date),
            "url": article_url
        })

        saved_count += 1

        print("본문 저장됨")

        # -----------------------------
        # 3-1. 댓글 수집
        # -----------------------------
        try:
            comments = get_article_comments()

            for comment in comments:
                comment_text = comment["comment_text"]
                comment_date = comment["comment_date"]

                # 댓글 날짜 못 가져오면 게시글 날짜로 대체
                if comment_date is None:
                    comment_date = article_date

                posts.append({
                    "source": "cafe",
                    "round": "38R_before_after",
                    "title": title,
                    "post_date": article_date,
                    "comment_date": comment_date,
                    "text_original": comment_text,
                    "text_clean": clean_text(comment_text),
                    "content_type": "cafe_comment",
                    "period": get_period(article_date),
                    "url": article_url
                })

                saved_count += 1

            print("댓글 저장됨:", len(comments))

        except Exception as e:
            print("댓글 수집 실패:", e)

    else:
        print("START_DATE~END_DATE 범위 밖 → 저장 안 함")


    checked_count = i + 1


    # -----------------------------
    # 4. 중간 저장
    # -----------------------------
    df = pd.DataFrame(posts)
    df = df.reindex(columns=FINAL_COLUMNS)
    df.to_csv(SAVE_FILE_CSV, index=False, encoding="utf-8-sig")
    df.to_excel(SAVE_FILE_XLSX, index=False)

    print("현재 저장 개수:", len(df))


    # -----------------------------
    # 5. 다음글 URL 가져오기
    # -----------------------------
    next_url = get_next_article_url()

    print("현재 article_url:", article_url)
    print("가져온 next_url:", next_url)
    print("같은 URL 여부:", article_url == next_url)

    if next_url is None:
        print("다음글 URL 없음 → 종료")
        break

    if next_url == article_url:
        print("문제 발생: 다음글 URL이 현재 글 URL과 같음")
        print("현재 article_url:", article_url)
        print("next_url:", next_url)
        break

    # 다음 반복에서 이동할 URL 저장
    article_url = next_url

    print("다음 반복에서 이동할 URL:", article_url)


# =============================
# 최종 저장
# =============================

df = pd.DataFrame(posts)
df = df.reindex(columns=FINAL_COLUMNS)

# 빈 본문 제거
df = df[df["text_clean"].notna()]
df = df[df["text_clean"].str.len() >= 2]

# 중복 제거
df = df.drop_duplicates(subset=["text_clean", "content_type", "url"])

df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce").dt.strftime("%Y-%m-%d %I:%M:%S %p")
df["comment_date"] = pd.to_datetime(df["comment_date"], errors="coerce").dt.strftime("%Y-%m-%d %I:%M:%S %p")

df.to_csv(SAVE_FILE_CSV, index=False, encoding="utf-8-sig")
df.to_excel(SAVE_FILE_XLSX, index=False)

print("\n수집 완료")
print("확인한 글 개수:", checked_count)
print("저장한 글 개수:", len(df))

df.head()

In [ ]:
df.shape

In [ ]:
def debug_date_candidates():
    global driver

    selectors = [
        ".WriterInfo .date",
        ".ArticleHeader .date",
        ".article_info .date",
        "span.date",
        ".date"
    ]

    for selector in selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            print("\nselector:", selector, "개수:", len(elems))

            for elem in elems:
                txt = elem.text.strip()
                if txt:
                    print("가져온 날짜 텍스트:", repr(txt), "->", parse_cafe_date(txt))

        except Exception as e:
            print(selector, "실패:", e)

In [ ]:
debug_date_candidates()
date_text, article_date = get_article_date()